# IEEE-CIS frozen-run validation
## tl;dr
Full input: 590,540 rows. All source/export hashes, required-field checks, 3,200 trial retention checks and 16 displayed HT estimates reconcile. Held-out declines: 5,158, including 3,558 legitimate payments. At 5% budget, weighted discovery precision is 80.20% versus 69.18% uniform, but BP RMSE is 5.21 versus 4.90 percentage points. These are controlled-censoring results, not production claims.
Executed 2026-09-03: all five Python cells ran top-to-bottom in the project environment through a lightweight cell runner (not a Jupyter kernel); actual aggregate stdout is saved below.
## Context & Methods
Companion audit for the full 2026-09-03 benchmark. Run from the repository root or docs directory with the project environment. Reads local data and sealed artifacts; never use this notebook in product/dashboard code. No data download, tuning, or external service.
### Key Assumptions
One historical transaction per TransactionID; TransactionDT is relative, not calendar time. isFraud is a perfect offline oracle only. No identity join. The frozen benchmark is conditional on a second censoring gate, not Razorpay production traffic. No outputs containing transaction rows should be committed.
## Data
Source: user-supplied official Kaggle train_transaction.csv.zip; extracted locally under data/raw/ieee-cis. Results: artifacts/ieee-full-2026-09-03.

In [1]:
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd

from blindspot.data.split import temporal_split

root = Path.cwd()
if not (root / "pyproject.toml").exists():
    root = root.parent
assert (root / "pyproject.toml").exists(), "Run from repository root or docs"
bundle = root / "artifacts/ieee-full-2026-09-03"
raw_path = root / "data/raw/ieee-cis/train_transaction.csv"
manifest = json.loads((bundle / "manifest.json").read_text())
benchmark = json.loads((bundle / "benchmark.json").read_text())


def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


assert sha256(raw_path) == manifest["source"]["files"][0]["sha256"]
checksums = json.loads((bundle / "checksums.json").read_text())
assert all(sha256(bundle / name) == digest for name, digest in checksums.items())
assert manifest["source"]["nrows"] is None
assert manifest["source"]["include_identity"] is False
print("Source hash and all seven export checksums match.")

Source hash and all seven export checksums match.


In [2]:
required = ["TransactionID", "TransactionDT", "TransactionAmt", "isFraud"]
header = pd.read_csv(raw_path, nrows=0)
frame = pd.read_csv(raw_path, usecols=required)
assert len(frame) == manifest["rows"]
assert frame.TransactionID.is_unique
assert not frame[required].isna().any().any()
assert np.isfinite(frame[required].to_numpy()).all()
assert frame.isFraud.isin([0, 1]).all()
assert frame.TransactionAmt.ge(0).all()
split = temporal_split(frame)
assert split.manifest() == manifest["split"]
profile = {
    "rows": len(frame),
    "columns": len(header.columns),
    "duplicate_ids": int(frame.TransactionID.duplicated().sum()),
    "required_nulls": int(frame.isna().sum().sum()),
    "fraud_rows": int(frame.isFraud.sum()),
    "time_min": int(frame.TransactionDT.min()),
    "time_max": int(frame.TransactionDT.max()),
}
print(json.dumps(profile, indent=2))
print(
    pd.DataFrame(
        [
            {
                "window": name,
                "rows": len(part),
                "fraud_rows": int(part.isFraud.sum()),
                "fraud_share": float(part.isFraud.mean()),
            }
            for name, part in [
                ("train", split.train),
                ("calibration", split.calibration),
                ("evaluation", split.evaluation),
            ]
        ]
    ).to_string(index=False)
)
features = pd.read_csv(raw_path, usecols=manifest["feature_columns"])
feature_nulls = features.isna().mean()
print(
    json.dumps(
        {
            "selected_features": len(features.columns),
            "features_with_missing": int(feature_nulls.gt(0).sum()),
            "highest_selected_feature_null_share": float(feature_nulls.max()),
        }
    )
)
del features

{
  "rows": 590540,
  "columns": 394,
  "duplicate_ids": 0,
  "required_nulls": 0,
  "fraud_rows": 20663,
  "time_min": 86400,
  "time_max": 15811131
}
     window   rows  fraud_rows  fraud_share
      train 413378       14538     0.035169
calibration  88581        3042     0.034341
 evaluation  88581        3083     0.034804
{"selected_features": 64, "features_with_missing": 48, "highest_selected_feature_null_share": 0.0005317167338368274}


## Results
### Independent full-window confusion counts
Use frozen decision IDs and raw held-out labels, not calibration metrics. PR-AUC and ROC-AUC for the full evaluation window cannot be reconstructed from decline-only scores and are not claimed here.

In [3]:
pool = pd.read_csv(bundle / "product/declines.csv")
truth = pd.read_csv(bundle / "sealed/truth.csv")
assert list(pool.columns) == [
    "row_id",
    "transaction_id",
    "transaction_dt",
    "transaction_amount",
    "risk_score",
    "decline_threshold",
]
assert pool.row_id.is_unique and truth.row_id.is_unique
assert set(pool.row_id) == set(truth.row_id)
assert set(pool.transaction_id).issubset(set(split.evaluation.TransactionID))
assert pool.risk_score.ge(manifest["decline_threshold"]).all()
evaluation = split.evaluation
declined = evaluation.TransactionID.isin(pool.transaction_id)
fraud = evaluation.isFraud.eq(1)
tp, fp = int((declined & fraud).sum()), int((declined & ~fraud).sum())
fn, tn = int((~declined & fraud).sum()), int((~declined & ~fraud).sum())
assert tp + fp == len(pool) == manifest["declines"]
assert fp == benchmark["oracle"]["false_declines"]
assert np.isclose(tp / len(pool), benchmark["oracle"]["block_precision"])
raw_outcomes = evaluation.set_index("TransactionID").loc[pool.transaction_id, "isFraud"].to_numpy()
sealed_outcomes = truth.set_index("row_id").loc[pool.row_id, "is_fraud"].to_numpy()
assert np.array_equal(raw_outcomes, sealed_outcomes)
assert np.isclose(
    evaluation.loc[declined & ~fraud, "TransactionAmt"].sum(),
    benchmark["oracle"]["false_decline_amount"],
)
heldout = {
    "true_positive": tp,
    "false_positive": fp,
    "false_negative": fn,
    "true_negative": tn,
    "precision": tp / (tp + fp),
    "recall": tp / (tp + fn),
    "false_positive_rate": fp / (fp + tn),
    "decline_rate": (tp + fp) / len(evaluation),
}
print(json.dumps(heldout, indent=2))

{
  "true_positive": 1600,
  "false_positive": 3558,
  "false_negative": 1483,
  "true_negative": 81940,
  "precision": 0.31019775106630476,
  "recall": 0.5189750243269543,
  "false_positive_rate": 0.04161500853821142,
  "decline_rate": 0.05822919136157867
}


In [4]:
runs = pd.read_csv(bundle / "runs.csv")
summary = pd.read_csv(bundle / "summary.csv")
registered = json.loads((bundle / "registered_design.json").read_text())
sweep = registered["sweep_config"]
assert sweep["repetitions"] == 200
assert len(runs) == 3200
assert set(runs.budget_rate) == set(sweep["budget_rates"])
assert set(runs.policy) == {"uniform", "margin_weighted"}
assert not runs[["budget_rate", "policy", "seed"]].duplicated().any()
for (rate, policy), group in runs.groupby(["budget_rate", "policy"]):
    assert set(group.seed) == set(range(sweep["seed_start"], sweep["seed_start"] + 200))
    row = summary.loc[(summary.budget_rate == rate) & (summary.policy == policy)].iloc[0]
    assert np.isclose(row.rmse_pp, np.sqrt(np.mean(group.error_pp**2)))
    assert np.isclose(row.coverage, group.covered.mean())
    assert np.isclose(row.stable_fraction, group.stable.mean())
    assert np.allclose(group.expected_budget, rate * len(pool))
public = json.loads((bundle / "public.json").read_text())
observations = json.loads((bundle / "observations.json").read_text())
for key, case in public["cases"].items():
    queue = pd.DataFrame(case["queue"])
    observed = pd.DataFrame(observations["cases"][key])
    assert set(queue.row_id) == set(observed.row_id)
    selected = queue.merge(observed, on="row_id", validate="one_to_one")
    ht = 1 - ((1 - selected.is_fraud) / selected.propensity).sum() / len(pool)
    assert np.isclose(ht, case["estimate"]["block_precision"])
print("All 3,200 trials retained; summary and 16 displayed HT estimates reconcile.")
print(
    summary.loc[
        summary.budget_rate <= 0.05,
        [
            "budget_rate",
            "policy",
            "expected_budget",
            "rmse_pp",
            "ci_width_pp_mean",
            "coverage",
            "stable_fraction",
            "fallback_fraction",
            "discovery_precision_mean",
            "discovery_recall_mean",
        ],
    ].to_string(index=False)
)
print(json.dumps(benchmark["paired_comparison"], indent=2))

All 3,200 trials retained; summary and 16 displayed HT estimates reconcile.
 budget_rate          policy  expected_budget   rmse_pp  ci_width_pp_mean  coverage  stable_fraction  fallback_fraction  discovery_precision_mean  discovery_recall_mean
      0.0025 margin_weighted           12.895 22.986303         72.544893     0.945            0.000               0.07                  0.791590               0.002858
      0.0025         uniform           12.895 22.624143         69.755741     0.935            0.000               0.01                  0.689411               0.002476
      0.0050 margin_weighted           25.790 15.983969         56.960823     0.935            0.000               0.01                  0.796524               0.005736
      0.0050         uniform           25.790 15.684266         55.881461     0.905            0.175               0.00                  0.683087               0.004900
      0.0100 margin_weighted           51.580 10.563737         44.342435     0

## Takeaways
Use the dated benchmark report for interpreted results. All comparisons are conditional on one frozen held-out population and 200 verification draws, not 200 independent training datasets. Stability flags do not certify nominal 95% coverage. Missing values in selected numeric features are handled by the frozen incumbent; feature timing and production evidence noise cannot be validated from this file alone.